In [ ]:
# Installing packages
install.packages("rio")
install.packages("RCurl")
install.packages("sqldf")
install.packages("tidyverse")
install.packages("dplyr")
install.packages("tibble")

# Loading the packages
library(rio)
library(RCurl)
library(sqldf)
library(tidyverse)
library(dplyr)
library(tibble)

In [ ]:
# Reading the cleaned datasets

hubs_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/hubs_clean.csv"
customers_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/customers_clean.csv"
drivers_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/drivers_clean.csv"
vehicles_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/vehicles_clean.csv"
orders_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/orders_clean.csv"
deliveries_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/deliveries_clean_valid.csv"
incidents_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/incidents_clean.csv"
complaints_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/complaints_clean.csv"
app_events_url <- "https://raw.githubusercontent.com/THEX3MN/NorthStar-Dataset/main/NorthStar%20Dataset%20(Cleaned)/app_events_clean.csv"

getURL(hubs_url)
getURL(customers_url)
getURL(drivers_url)
getURL(vehicles_url)
getURL(orders_url)
getURL(deliveries_url)
getURL(incidents_url)
getURL(complaints_url)
getURL(app_events_url)

hubs <- import(hubs_url)
customers <- import(customers_url)
drivers <- import(drivers_url)
vehicles <- import(vehicles_url)
orders <- import(orders_url)
deliveries <- import(deliveries_url)
incidents <- import(incidents_url)
complaints <- import(complaints_url)
app_events <- import(app_events_url)

# Checking that the datasets have loaded correctly

head(hubs)
head(customers)
head(drivers)
head(vehicles)
head(orders)
head(deliveries)
head(incidents)
head(complaints)
head(app_events)

In [ ]:
# Descriptive Statistics

# Hubs summary
summary(hubs)

# Customers summary
summary(customers)

# Drivers summary
summary(drivers)

# Vehicles summary
summary(vehicles)

# Orders summary
summary(orders)

# Deliveries summary
summary(deliveries)

# Incidents summary
summary(incidents)

# Complaints summary
summary(complaints)

# App events summary
summary(app_events)

# Business summary tables

business_summary <- tibble(
  metric = c(
    "Average order value",
    "Average delivery distance",
    "Average delivery rating",
    "Average incident resolution time",
    "Average complaint resolution time",
    "Average fuel or charge cost",
    "Delay rate (%)",
    "Failure rate (%)",
    "Total complaints",
    "Total incidents"
  ),
  value = c(
    round(mean(orders$order_value, na.rm = TRUE), 2),
    round(mean(deliveries$route_distance_km, na.rm = TRUE), 2),
    round(mean(deliveries$customer_rating_post_delivery, na.rm = TRUE), 2),
    round(mean(incidents$resolved_hours, na.rm = TRUE), 2),
    round(mean(complaints$resolution_days, na.rm = TRUE), 2),
    round(mean(deliveries$fuel_or_charge_cost, na.rm = TRUE), 2),
    round(mean(deliveries$delivery_status == "Delayed", na.rm = TRUE) * 100, 2),
    round(mean(deliveries$delivery_status == "Failed", na.rm = TRUE) * 100, 2),
    nrow(complaints),
    nrow(incidents)
  )
)

business_summary

In [ ]:
# Grouped Statistical Analysis

# By Zone

zone_analysis <- deliveries %>%
  left_join(hubs, by = "hub_id") %>%
  group_by(zone) %>%
  summarise(
    total_deliveries = n(),
    avg_customer_rating = round(mean(customer_rating_post_delivery, na.rm = TRUE), 2),
    avg_fuel_or_charge_cost = round(mean(fuel_or_charge_cost, na.rm = TRUE), 2),
    avg_manual_overrides = round(mean(manual_route_override_count, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_deliveries))

zone_analysis

# By Hub

hub_analysis <- deliveries %>%
  left_join(hubs, by = "hub_id") %>%
  group_by(hub_id, hub_name, zone, hub_type, capacity_score) %>%
  summarise(
    total_deliveries = n(),
    ontime_deliveries = sum(delivery_status == "Ontime", na.rm = TRUE),
    delayed_deliveries = sum(delivery_status == "Delayed", na.rm = TRUE),
    failed_deliveries = sum(delivery_status == "Failed", na.rm = TRUE),
    avg_customer_rating = round(mean(customer_rating_post_delivery, na.rm = TRUE), 2),
    avg_manual_overrides = round(mean(manual_route_override_count, na.rm = TRUE), 2),
    avg_fuel_or_charge_cost = round(mean(fuel_or_charge_cost, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_deliveries))

hub_analysis

# By Service Type

service_type_analysis <- orders %>%
  group_by(service_type) %>%
  summarise(
    total_orders = n(),
    avg_order_value = round(mean(order_value, na.rm = TRUE), 2),
    avg_promised_window = round(mean(promised_window_hours, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_orders))

service_type_analysis

# By Driver Employment Type

driver_by_id <- deliveries %>%
  left_join(drivers, by = "driver_id") %>%
  group_by(driver_id, employment_type, driver_rating) %>%
  summarise(
    total_deliveries = n(),
    total_customer_rating = sum(customer_rating_post_delivery, na.rm = TRUE),
    avg_customer_rating = round(mean(customer_rating_post_delivery, na.rm = TRUE), 2),
    ontime_deliveries = sum(delivery_status == "Ontime", na.rm = TRUE),
    delayed_deliveries = sum(delivery_status == "Delayed", na.rm = TRUE),
    failed_deliveries = sum(delivery_status == "Failed", na.rm = TRUE),
    .groups = "drop"
  )

driver_type_analysis <- driver_by_id %>%
  group_by(employment_type) %>%
  summarise(
    total_drivers = n_distinct(driver_id),
    total_deliveries = sum(total_deliveries, na.rm = TRUE),
    avg_driver_rating = round(mean(driver_rating, na.rm = TRUE), 2),
    avg_customer_rating = round(sum(total_customer_rating, na.rm = TRUE) / sum(total_deliveries, na.rm = TRUE), 2),
    ontime_deliveries = sum(ontime_deliveries, na.rm = TRUE),
    delayed_deliveries = sum(delayed_deliveries, na.rm = TRUE),
    failed_deliveries = sum(failed_deliveries, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(total_deliveries))

driver_type_analysis

# By Vehicle Maintenance Status

vehicle_by_id <- deliveries %>%
  left_join(vehicles, by = "vehicle_id") %>%
  group_by(vehicle_id, maintenance_status, battery_health_pct) %>%
  summarise(
    total_deliveries = n(),
    total_customer_rating = sum(customer_rating_post_delivery, na.rm = TRUE),
    total_fuel_or_charge_cost = sum(fuel_or_charge_cost, na.rm = TRUE),
    avg_customer_rating = round(mean(customer_rating_post_delivery, na.rm = TRUE), 2),
    avg_fuel_or_charge_cost = round(mean(fuel_or_charge_cost, na.rm = TRUE), 2),
    ontime_deliveries = sum(delivery_status == "Ontime", na.rm = TRUE),
    delayed_deliveries = sum(delivery_status == "Delayed", na.rm = TRUE),
    failed_deliveries = sum(delivery_status == "Failed", na.rm = TRUE),
    .groups = "drop"
  )

vehicle_maintenance_analysis <- vehicle_by_id %>%
  group_by(maintenance_status) %>%
  summarise(
    total_vehicles = n_distinct(vehicle_id),
    total_deliveries = sum(total_deliveries, na.rm = TRUE),
    avg_battery_health_pct = round(mean(battery_health_pct, na.rm = TRUE), 2),
    avg_customer_rating = round(sum(total_customer_rating, na.rm = TRUE) / sum(total_deliveries, na.rm = TRUE), 2),
    avg_fuel_or_charge_cost = round(sum(total_fuel_or_charge_cost, na.rm = TRUE) / sum(total_deliveries, na.rm = TRUE), 2),
    ontime_deliveries = sum(ontime_deliveries, na.rm = TRUE),
    delayed_deliveries = sum(delayed_deliveries, na.rm = TRUE),
    failed_deliveries = sum(failed_deliveries, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(total_deliveries))

vehicle_maintenance_analysis

# By Customer Type

customer_by_id <- customers %>%
  left_join(orders, by = "customer_id") %>%
  group_by(customer_id, customer_type, loyalty_score, app_engagement_score) %>%
  summarise(
    total_orders = sum(!is.na(order_id)),
    total_order_value = sum(order_value, na.rm = TRUE),
    .groups = "drop"
  )

customer_type_analysis <- customer_by_id %>%
  group_by(customer_type) %>%
  summarise(
    total_customers = n_distinct(customer_id),
    total_orders = sum(total_orders, na.rm = TRUE),
    total_order_value = sum(total_order_value, na.rm = TRUE),
    avg_loyalty_score = round(mean(loyalty_score, na.rm = TRUE), 2),
    avg_app_engagement_score = round(mean(app_engagement_score, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_orders))

customer_type_analysis

# By Complaint Severity

complaint_severity_analysis <- complaints %>%
  group_by(severity) %>%
  summarise(
    total_complaints = n(),
    avg_resolution_days = round(mean(resolution_days, na.rm = TRUE), 2),
    avg_compensation_amount = round(mean(compensation_amount, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_complaints))

complaint_severity_analysis

# By Complaint Type

complaint_type_analysis <- complaints %>%
  group_by(complaint_type) %>%
  summarise(
    total_complaints = n(),
    avg_resolution_days = round(mean(resolution_days, na.rm = TRUE), 2),
    avg_compensation_amount = round(mean(compensation_amount, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_complaints))

complaint_type_analysis

# By Complaint Channel

complaint_channel_analysis <- complaints %>%
  group_by(channel) %>%
  summarise(
    total_complaints = n(),
    avg_resolution_days = round(mean(resolution_days, na.rm = TRUE), 2),
    avg_compensation_amount = round(mean(compensation_amount, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_complaints))

complaint_channel_analysis

# By Incident Severity

incident_severity_analysis <- incidents %>%
  group_by(severity) %>%
  summarise(
    total_incidents = n(),
    avg_resolution_hours = round(mean(resolved_hours, na.rm = TRUE), 2),
    .groups = "drop"
  ) %>%
  arrange(desc(total_incidents))

incident_severity_analysis

# By App Event Type

app_event_type_analysis <- app_events %>%
  group_by(event_type) %>%
  summarise(
    total_events = n(),
    avg_api_latency_ms = round(mean(api_latency_ms, na.rm = TRUE), 2),
    success_events = sum(success_flag == 1, na.rm = TRUE),
    failed_events = sum(success_flag == 0, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(total_events))

app_event_type_analysis

In [ ]:
# Correlation and Relationship Analysis

delivery_distance_rating <- deliveries[, c("route_distance_km", "customer_rating_post_delivery")]
delivery_distance_rating <- delivery_distance_rating[complete.cases(delivery_distance_rating), ]

delivery_overrides_cost <- deliveries[, c("manual_route_override_count", "fuel_or_charge_cost")]
delivery_overrides_cost <- delivery_overrides_cost[complete.cases(delivery_overrides_cost), ]

driver_delivery_relationship <- sqldf("
SELECT
    dr.driver_rating,
    d.customer_rating_post_delivery
FROM deliveries d
LEFT JOIN drivers dr
ON d.driver_id = dr.driver_id
WHERE dr.driver_rating IS NOT NULL
AND d.customer_rating_post_delivery IS NOT NULL
")

vehicle_delivery_relationship <- sqldf("
SELECT
    v.battery_health_pct,
    CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END AS failure_flag
FROM deliveries d
LEFT JOIN vehicles v
ON d.vehicle_id = v.vehicle_id
WHERE v.battery_health_pct IS NOT NULL
")

app_latency_success <- app_events[, c("api_latency_ms", "success_flag")]
app_latency_success <- app_latency_success[complete.cases(app_latency_success), ]

complaint_resolution_compensation <- complaints[, c("resolution_days", "compensation_amount")]
complaint_resolution_compensation <- complaint_resolution_compensation[complete.cases(complaint_resolution_compensation), ]

# Delivery distance and customer rating

cor.test(
    delivery_distance_rating$route_distance_km,
    delivery_distance_rating$customer_rating_post_delivery
)

ggplot(delivery_distance_rating,
       aes(x = route_distance_km,
           y = customer_rating_post_delivery)) +
    geom_point() +
    labs(
        title = "Route Distance vs Customer Rating",
        x = "Route Distance (km)",
        y = "Customer Rating"
    )

# Manual route overrides and fuel or charge cost

cor.test(
    delivery_overrides_cost$manual_route_override_count,
    delivery_overrides_cost$fuel_or_charge_cost
)

ggplot(delivery_overrides_cost,
       aes(x = manual_route_override_count,
           y = fuel_or_charge_cost)) +
    geom_point() +
    labs(
        title = "Manual Route Overrides vs Fuel or Charge Cost",
        x = "Manual Route Override Count",
        y = "Fuel or Charge Cost"
    )

# Driver rating and customer rating

cor.test(
    driver_delivery_relationship$driver_rating,
    driver_delivery_relationship$customer_rating_post_delivery
)

ggplot(driver_delivery_relationship,
       aes(x = driver_rating,
           y = customer_rating_post_delivery)) +
    geom_point() +
    labs(
        title = "Driver Rating vs Customer Rating",
        x = "Driver Rating",
        y = "Customer Rating"
    )

# Battery health and delivery failure

cor.test(
    vehicle_delivery_relationship$battery_health_pct,
    vehicle_delivery_relationship$failure_flag
)

ggplot(vehicle_delivery_relationship,
       aes(x = battery_health_pct,
           y = failure_flag)) +
    geom_point() +
    labs(
        title = "Battery Health vs Delivery Failure",
        x = "Battery Health (%)",
        y = "Failure Flag"
    )

# API latency and success flag

cor.test(
    app_latency_success$api_latency_ms,
    app_latency_success$success_flag
)

ggplot(app_latency_success,
       aes(x = api_latency_ms,
           y = success_flag)) +
    geom_point() +
    labs(
        title = "API Latency vs Success Flag",
        x = "API Latency (ms)",
        y = "Success Flag"
    )

# Complaint resolution days and compensation amount

cor.test(
    complaint_resolution_compensation$resolution_days,
    complaint_resolution_compensation$compensation_amount
)

ggplot(complaint_resolution_compensation,
       aes(x = resolution_days,
           y = compensation_amount)) +
    geom_point() +
    labs(
        title = "Complaint Resolution Days vs Compensation Amount",
        x = "Resolution Days",
        y = "Compensation Amount"
    )

# Delivery status distribution

delivery_status_chart <- sqldf("
SELECT
    delivery_status,
    COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
")

ggplot(delivery_status_chart,
       aes(x = delivery_status,
           y = total_deliveries)) +
    geom_bar(stat = 'identity') +
    labs(
        title = "Delivery Status Distribution",
        x = "Delivery Status",
        y = "Total Deliveries"
    )

# Complaint severity distribution

complaint_severity_chart <- sqldf("
SELECT
    severity,
    COUNT(*) AS total_complaints
FROM complaints
GROUP BY severity
")

ggplot(complaint_severity_chart,
       aes(x = severity,
           y = total_complaints)) +
    geom_bar(stat = 'identity') +
    labs(
        title = "Complaint Severity Distribution",
        x = "Complaint Severity",
        y = "Total Complaints"
    )

# Service type distribution

service_type_chart <- sqldf("
SELECT
    service_type,
    COUNT(*) AS total_orders
FROM orders
GROUP BY service_type
")

ggplot(service_type_chart,
       aes(x = service_type,
           y = total_orders)) +
    geom_bar(stat = 'identity') +
    labs(
        title = "Service Type Distribution",
        x = "Service Type",
        y = "Total Orders"
    )

# Delivery status pie chart

delivery_status_pie <- sqldf("
SELECT
    delivery_status,
    COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
")

ggplot(delivery_status_pie,
       aes(x = '',
           y = total_deliveries,
           fill = delivery_status)) +
    geom_bar(stat = 'identity', width = 1) +
    coord_polar('y', start = 0) +
    labs(
        title = "Delivery Status Share"
    )

# Matching orders and deliveries

matched_data <- merge(
    orders,
    deliveries,
    by = "order_id"
)

In [ ]:
# Visualisations

# 1) Deliveries by Zone
ggplot(zone_analysis, aes(x = zone, y = total_deliveries)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Deliveries by Zone",
        x = "Zone",
        y = "Total Deliveries"
    )

# 2) Orders by Service Type
ggplot(service_type_analysis, aes(x = service_type, y = total_orders)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Orders by Service Type",
        x = "Service Type",
        y = "Total Orders"
    )

# 3) Complaints by Severity
ggplot(complaint_severity_analysis, aes(x = severity, y = total_complaints)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Complaints by Severity",
        x = "Severity",
        y = "Total Complaints"
    )

# 4) Complaint Severity Pie Chart
ggplot(complaint_severity_analysis, aes(x = "", y = total_complaints, fill = severity)) +
    geom_bar(stat = "identity", width = 1) +
    coord_polar("y", start = 0) +
    labs(
        title = "Complaint Severity Share"
    )

# 5) Incidents by Severity
ggplot(incident_severity_analysis, aes(x = severity, y = total_incidents)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Incidents by Severity",
        x = "Severity",
        y = "Total Incidents"
    )

# 6) Customer Type vs Orders
ggplot(customer_type_analysis, aes(x = customer_type, y = total_orders)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Orders by Customer Type",
        x = "Customer Type",
        y = "Total Orders"
    )

# 7) Driver Employment Type vs Deliveries
ggplot(driver_type_analysis, aes(x = employment_type, y = total_deliveries)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Deliveries by Driver Employment Type",
        x = "Employment Type",
        y = "Total Deliveries"
    )

# 8) Vehicle Maintenance Status vs Deliveries
ggplot(vehicle_maintenance_analysis, aes(x = maintenance_status, y = total_deliveries)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total Deliveries by Vehicle Maintenance Status",
        x = "Maintenance Status",
        y = "Total Deliveries"
    )

# 9) App Events by Event Type
ggplot(app_event_type_analysis, aes(x = event_type, y = total_events)) +
    geom_bar(stat = "identity") +
    labs(
        title = "Total App Events by Event Type",
        x = "Event Type",
        y = "Total Events"
    )

# 10) Route Distance vs Customer Rating
ggplot(deliveries, aes(x = route_distance_km, y = customer_rating_post_delivery)) +
    geom_point() +
    labs(
        title = "Route Distance vs Customer Rating",
        x = "Route Distance (km)",
        y = "Customer Rating"
    )

# 11) Manual Route Overrides vs Fuel or Charge Cost
ggplot(deliveries, aes(x = manual_route_override_count, y = fuel_or_charge_cost)) +
    geom_point() +
    labs(
        title = "Manual Route Overrides vs Fuel or Charge Cost",
        x = "Manual Route Override Count",
        y = "Fuel or Charge Cost"
    )

# 12) Battery Health vs Customer Rating
vehicle_delivery_data <- merge(deliveries, vehicles, by = "vehicle_id")

ggplot(vehicle_delivery_data, aes(x = battery_health_pct, y = customer_rating_post_delivery)) +
    geom_point() +
    labs(
        title = "Battery Health vs Customer Rating",
        x = "Battery Health (%)",
        y = "Customer Rating"
    )

# 13) API Latency vs Success Flag
ggplot(app_events, aes(x = api_latency_ms, y = success_flag)) +
    geom_point() +
    labs(
        title = "API Latency vs Success Flag",
        x = "API Latency (ms)",
        y = "Success Flag"
    )